# 01 — Data Cleaning and Quality Validation

## Project
Netflix Customer Intelligence & AI Analytics Copilot

## Objective
Inspect and clean the nine source datasets before analytics and modelling.

This notebook documents and executes the existing cleaning implementation in
`src/data/clean_data.py`. It does not replace or silently modify that pipeline.

## Cleaning operations
- Normalize surrounding whitespace in text columns.
- Convert empty strings to missing values.
- Remove exact duplicate rows from the customer dataset only.
- Mark customer ages outside 0–120 as missing.
- Parse configured date columns.
- Report duplicate identifiers without automatically deleting them.
- Validate selected numeric ranges.
- Save processed CSV files and a cleaning report.

**Important:** Numeric validation findings are reported; the pipeline does not
automatically remove or correct every invalid numeric value.


## 1. Imports and project paths

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "data" / "clean_data.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "data" / "reports"

import os
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)
print("Raw data directory exists:", RAW_DIR.exists())
print("Raw CSV count:", len(list(RAW_DIR.glob("*.csv"))))


Project root: /workspaces/Netflix-Customer-Intelligence
Raw data directory exists: True
Raw CSV count: 9


## 2. Inspect raw datasets

In [2]:
raw_files = sorted(RAW_DIR.glob("*.csv"))
if len(raw_files) != 9:
    raise ValueError(f"Expected 9 raw CSV files, found {len(raw_files)}")

raw_summary = []
for file in raw_files:
    df = pd.read_csv(file, low_memory=False)
    raw_summary.append({
        "file": file.name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum())
    })

raw_summary = pd.DataFrame(raw_summary)
display(raw_summary)


,file,rows,columns,missing_cells,duplicate_rows
0,churn_labels.csv,8000,4,12018,0
1,content.csv,500,9,0,0
2,customer_feedback.csv,5046,6,0,0
3,customers.csv,8048,12,9517,48
4,payments.csv,110735,8,214264,0
5,subscription_plans.csv,3,6,0,0
6,subscriptions.csv,8875,10,21768,0
7,support_tickets.csv,6120,11,3056,0
8,viewing_activity.csv,66422,9,1195,0


## 3. Execute the project cleaning pipeline

The next cell runs the existing source script. It writes cleaned datasets to
`data/processed/` and the report to `data/reports/cleaning_report.csv`.


In [3]:
%run src/data/clean_data.py

CLEANING PIPELINE COMPLETED
Files processed: 9
Cleaned data: data/processed
Cleaning report: data/reports/cleaning_report.csv
                  file  rows_before  rows_after  rows_removed  exact_duplicates_removed  invalid_ages_marked_missing                                                            invalid_dates_by_column          duplicate_ids                                                                                             validation_checks  missing_cells_after
      churn_labels.csv         8000        8000             0                         0                            0                                                                  {'churn_date': 0}     {'customer_id': 0}                                                                                                            {}                12018
           content.csv          500         500             0                         0                            0                                                  

## 4. Review the cleaning report

In [4]:
cleaning_report_path = REPORT_DIR / "cleaning_report.csv"
cleaning_report = pd.read_csv(cleaning_report_path)
display(cleaning_report)


,file,rows_before,rows_after,rows_removed,exact_duplicates_removed,invalid_ages_marked_missing,invalid_dates_by_column,duplicate_ids,validation_checks,missing_cells_after
0,churn_labels.csv,8000,8000,0,0,0,{'churn_date': 0},{'customer_id': 0},{},12018
1,content.csv,500,500,0,0,0,{},{'content_id': 0},{},0
2,customer_feedback.csv,5046,5046,0,0,0,{'feedback_date': 0},{'feedback_id': 0},{'rating_outside_1_5': 0},0
3,customers.csv,8048,8000,48,48,8,{'registration_date': 0},{'customer_id': 0},{},9472
4,payments.csv,110735,110735,0,0,0,{'payment_date': 0},{'payment_id': 0},{'negative_payment_amount': 0},214264
5,subscription_plans.csv,3,3,0,0,0,{},{'plan_id': 0},{},0
6,subscriptions.csv,8875,8875,0,0,0,"{'subscription_start_date': 0, 'subscription_e...",{'subscription_id': 0},{},21768
7,support_tickets.csv,6120,6120,0,0,0,{'ticket_date': 61},{'ticket_id': 0},"{'satisfaction_outside_1_5': 0, 'negative_reso...",3117
8,viewing_activity.csv,66422,66422,0,0,0,{'viewing_date': 0},{'viewing_id': 0},"{'completion_outside_0_100': 0, 'negative_watc...",1195


## 5. Compare raw and processed datasets

In [5]:
comparison = []
for file in raw_files:
    raw = pd.read_csv(file, low_memory=False)
    cleaned = pd.read_csv(PROCESSED_DIR / file.name, low_memory=False)
    comparison.append({
        "file": file.name,
        "raw_rows": len(raw),
        "processed_rows": len(cleaned),
        "rows_removed": len(raw) - len(cleaned),
        "raw_columns": len(raw.columns),
        "processed_columns": len(cleaned.columns),
        "processed_missing_cells": int(cleaned.isna().sum().sum())
    })

comparison = pd.DataFrame(comparison)
display(comparison)


,file,raw_rows,processed_rows,rows_removed,raw_columns,processed_columns,processed_missing_cells
0,churn_labels.csv,8000,8000,0,4,4,12018
1,content.csv,500,500,0,9,9,0
2,customer_feedback.csv,5046,5046,0,6,6,0
3,customers.csv,8048,8000,48,12,12,9472
4,payments.csv,110735,110735,0,8,8,214264
5,subscription_plans.csv,3,3,0,6,6,0
6,subscriptions.csv,8875,8875,0,10,10,21768
7,support_tickets.csv,6120,6120,0,11,11,3117
8,viewing_activity.csv,66422,66422,0,9,9,1195


## 6. Data-cleaning conclusions and limitations

- The pipeline processes all nine expected datasets.
- Exact duplicate removal is deliberately restricted to customer records.
- Duplicate identifiers in other datasets are reported for investigation.
- Implausible ages are marked missing rather than imputed.
- Malformed dates are converted to `NaT`.
- Numeric validation checks report issues but do not universally repair them.
- Relationship integrity is assessed separately by the project's relationship
  validation script.

The generated cleaning report should be reviewed before downstream modelling.
